# StockAI Portal — Run & Test on Google Colab

This notebook lets you **upload the project zip directly** (no GitHub needed) and get a **temporary public link** so you can click and use the live site right from Colab.

**Run the cells in order.** Cell 2 will pop up a file picker — choose your `STOCKS_ANALYSIS_with_Portal.zip` (or any zip that contains a `StockAI_Portal` folder). Cell 4 will ask you to paste your free ngrok authtoken — get one (1 minute, free) at https://dashboard.ngrok.com/get-started/your-authtoken after signing up at https://dashboard.ngrok.com/signup

In [ ]:
!pip install -q flask yfinance pandas numpy scipy statsmodels reportlab python-docx opencv-python-headless pyngrok deep-translator pdfplumber matplotlib mplfinance

In [ ]:
# --- Upload your project zip directly (recommended, no GitHub needed) ---
from google.colab import files
import zipfile, os, glob

print("Choose your project zip file (e.g. STOCKS_ANALYSIS_with_Portal.zip)...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_dir = "/content/extracted"
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(extract_dir)

matches = glob.glob(f"{extract_dir}/**/StockAI_Portal", recursive=True)
if not matches:
    raise FileNotFoundError(
        "Could not find a 'StockAI_Portal' folder inside the uploaded zip. "
        "Make sure you uploaded the full project zip, not just a single file."
    )
portal_dir = matches[0]
print("Found the app at:", portal_dir)
%cd $portal_dir

### Alternative: clone from GitHub instead of uploading
Only use this if you've already pushed the **extracted** project (not the zip file itself) to GitHub. Skip Cell 2 above if you use this instead.
```python
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
import os
!git clone $REPO_URL /content/repo
%cd /content/repo/StockAI_Portal
```

In [ ]:
# --- Paste your free ngrok authtoken when prompted (input is hidden) ---
from pyngrok import ngrok
from getpass import getpass

print("Get a free token at: https://dashboard.ngrok.com/get-started/your-authtoken")
NGROK_AUTH_TOKEN = getpass("Paste your ngrok authtoken and press Enter: ")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
import threading
from app import app

def run_app():
    app.run(port=5000, use_reloader=False)

thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

public_url = ngrok.connect(5000)
print("Your live StockAI Portal link (click / share this):")
print(public_url)

### Notes
- This link works **only while this Colab notebook keeps running** — it's for quick testing/demo, not permanent hosting.
- Colab sessions time out after inactivity, which closes the tunnel — re-run the cells to get a fresh link.
- For a permanent 24/7 live link, deploy to Render/Railway instead (see the main README's "Get a permanent live link" section) — same `app.py`, no code changes needed.
- To stop the tunnel manually: `ngrok.kill()`
- Made a change locally and want to re-test? Just re-run from Cell 2 with your updated zip — no need to restart the whole notebook.